# `DocLayNet` DeepLabV3-MobileNetV2 Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` code to recreate a
**DeepLabV3** architecture with a **MobileNetV2** backbone from scratch
(~**5.1M** parameters)!

- DeepLabV3 paper: https://arxiv.org/abs/1706.05587
- MobileNetV2 paper: https://arxiv.org/abs/1801.04381

The architecture depends on 3 components:
1. A `MobileNetV2Encoder` backbone with **output stride 16**: the last stride-2
   stage keeps stride 1 and uses **dilation 2** instead, so the deepest feature
   stays at 1/16 resolution.
2. An **ASPP (Atrous Spatial Pyramid Pooling)** head: parallel `1x1` and dilated
   `3x3` convolutions (rates 6, 12, 18) plus global image pooling.
3. A classifier (`3x3` refine + `1x1` projection) and 16x bilinear upsampling.


## 1. The `MobileNetV2` encoder backbone (from scratch)

We first rebuild the **MobileNetV2** feature extractor from the original paper
(https://arxiv.org/abs/1801.04381), exactly as in `src/models/backbones.py`.
Its core component is the **inverted residual block** (`1x1` expansion ->
`3x3` depthwise -> `1x1` linear projection), stacked with the paper's Table 2
configuration `(t, c, n, s)`:

```
(1, 16, 1, 1), (6, 24, 2, 2), (6, 32, 3, 2),
(6, 64, 4, 2), (6, 96, 3, 1), (6, 160, 3, 2), (6, 320, 1, 1)
```

For an input `(B, 3, 512, 512)` the encoder returns 5 multi-scale feature maps:

```python
x  -> stem + stage1 -> f1 (B,  16, 256, 256)   # 1/2
f1 -> stage2        -> f2 (B,  24, 128, 128)   # 1/4
f2 -> stage3        -> f3 (B,  32,  64,  64)   # 1/8
f3 -> stage4        -> f4 (B,  96,  32,  32)   # 1/16
f4 -> stage5        -> f5 (B, 320,  16,  16)   # 1/32 (or 1/16 when dilated)
```

We start with a shared `ConvBNReLU` helper block used across all our benchmark architectures.


In [ ]:
# Let's import all necessary modules for this architecture
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBNReLU(nn.Module):
    '''
    Standard Conv2d -> BatchNorm2d -> ReLU block used across all architectures.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        kernel_size (int): convolution kernel size.
        stride (int): convolution stride.
        padding (int): convolution padding.
        dilation (int): convolution dilation.
        groups (int): convolution groups (used for depthwise convolutions).
        relu6 (bool): if True, uses ReLU6 (MobileNetV2 convention) instead of ReLU.
    '''

    def __init__(self, m: int, n: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, dilation: int = 1, groups: int = 1,
                 relu6: bool = False) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=kernel_size,
                      stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU6() if relu6 else nn.ReLU()
        )

    def forward(self, x) -> torch.Tensor:
        return self.block(x)


In [ ]:
class InvertedResidual(nn.Module):
    '''
    Class that defines the MobileNetV2 inverted residual block:
    1x1 expansion -> 3x3 depthwise -> 1x1 linear projection.
    Reference: https://arxiv.org/abs/1801.04381

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        stride (int): stride of the depthwise convolution.
        expand_ratio (int): channel expansion factor t of the paper.
        dilation (int): dilation of the depthwise convolution (used to keep
            spatial resolution for DeepLabV3, output stride 16).
    '''

    def __init__(self, m: int, n: int, stride: int = 1,
                 expand_ratio: int = 6, dilation: int = 1) -> None:
        super().__init__()
        hidden = m * expand_ratio
        self.use_residual = (stride == 1 and m == n)

        layers = []
        # 1x1 pointwise expansion (skipped when t=1, first bottleneck)
        if expand_ratio != 1:
            layers.append(ConvBNReLU(m=m, n=hidden, kernel_size=1, stride=1,
                                     padding=0, relu6=True))
        # 3x3 depthwise convolution (groups = hidden channels)
        layers.append(ConvBNReLU(m=hidden, n=hidden, kernel_size=3, stride=stride,
                                 padding=dilation, dilation=dilation,
                                 groups=hidden, relu6=True))
        # 1x1 linear projection (no activation, "linear bottleneck")
        layers.append(nn.Conv2d(in_channels=hidden, out_channels=n,
                                kernel_size=(1, 1), stride=(1, 1),
                                padding=(0, 0), bias=False))
        layers.append(nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                                     affine=True, track_running_stats=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x) -> torch.Tensor:
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)


In [ ]:
class MobileNetV2Encoder(nn.Module):
    '''
    Class that defines the full MobileNetV2 feature-extractor backbone from
    scratch (no classification head), returning multi-scale feature maps.
    Reference: https://arxiv.org/abs/1801.04381

    Bottleneck configuration (t, c, n, s) follows Table 2 of the paper:
        (1, 16, 1, 1), (6, 24, 2, 2), (6, 32, 3, 2),
        (6, 64, 4, 2), (6, 96, 3, 1), (6, 160, 3, 2), (6, 320, 1, 1)

    Feature maps returned for an input of shape (B, Cin, H, W):
        f1: (B,  16, H/2,  W/2)
        f2: (B,  24, H/4,  W/4)
        f3: (B,  32, H/8,  W/8)
        f4: (B,  96, H/16, W/16)
        f5: (B, 320, H/32, W/32)  (H/16 when output_stride=16)

    Args:
        Cin (int): number of input channels (3 for RGB document images).
        output_stride (int): 32 for standard encoders (U-Net), 16 for
            DeepLabV3 (last downsampling replaced by dilation=2).
        include_top_conv (bool): if True, appends the final 1x1 convolution
            to 1280 channels of the original paper on top of f5 (used by the
            U-Net variant as a wide bottleneck).
    '''

    def __init__(self, Cin: int = 3, output_stride: int = 32,
                 include_top_conv: bool = False) -> None:
        super().__init__()
        assert output_stride in [16, 32], "output_stride must be 16 or 32"

        # When output_stride=16, the last stride-2 stage keeps stride 1
        # and uses dilation 2 instead (DeepLabV3 convention)
        last_stride = 2 if output_stride == 32 else 1
        last_dilation = 1 if output_stride == 32 else 2

        # Stem: 3x3/2 convolution to 32 channels
        self.stem = ConvBNReLU(m=Cin, n=32, kernel_size=3, stride=2,
                               padding=1, relu6=True)

        # Stage 1 -> f1 at 1/2 resolution (t=1, c=16, n=1, s=1)
        self.stage1 = InvertedResidual(m=32, n=16, stride=1, expand_ratio=1)

        # Stage 2 -> f2 at 1/4 resolution (t=6, c=24, n=2, s=2)
        self.stage2 = nn.Sequential(
            InvertedResidual(m=16, n=24, stride=2, expand_ratio=6),
            InvertedResidual(m=24, n=24, stride=1, expand_ratio=6)
        )

        # Stage 3 -> f3 at 1/8 resolution (t=6, c=32, n=3, s=2)
        self.stage3 = nn.Sequential(
            InvertedResidual(m=24, n=32, stride=2, expand_ratio=6),
            InvertedResidual(m=32, n=32, stride=1, expand_ratio=6),
            InvertedResidual(m=32, n=32, stride=1, expand_ratio=6)
        )

        # Stage 4 -> f4 at 1/16 resolution (t=6, c=64, n=4, s=2) + (t=6, c=96, n=3, s=1)
        self.stage4 = nn.Sequential(
            InvertedResidual(m=32, n=64, stride=2, expand_ratio=6),
            InvertedResidual(m=64, n=64, stride=1, expand_ratio=6),
            InvertedResidual(m=64, n=64, stride=1, expand_ratio=6),
            InvertedResidual(m=64, n=64, stride=1, expand_ratio=6),
            InvertedResidual(m=64, n=96, stride=1, expand_ratio=6),
            InvertedResidual(m=96, n=96, stride=1, expand_ratio=6),
            InvertedResidual(m=96, n=96, stride=1, expand_ratio=6)
        )

        # Stage 5 -> f5 at 1/32 (or dilated 1/16) resolution
        # (t=6, c=160, n=3, s=2) + (t=6, c=320, n=1, s=1)
        self.stage5 = nn.Sequential(
            InvertedResidual(m=96, n=160, stride=last_stride,
                             expand_ratio=6, dilation=last_dilation),
            InvertedResidual(m=160, n=160, stride=1,
                             expand_ratio=6, dilation=last_dilation),
            InvertedResidual(m=160, n=160, stride=1,
                             expand_ratio=6, dilation=last_dilation),
            InvertedResidual(m=160, n=320, stride=1,
                             expand_ratio=6, dilation=last_dilation)
        )

        # Optional final 1x1 convolution to 1280 channels (paper's last layer)
        if include_top_conv:
            self.top_conv = ConvBNReLU(m=320, n=1280, kernel_size=1,
                                       stride=1, padding=0, relu6=True)
            self.out_channels = [16, 24, 32, 96, 1280]
        else:
            self.top_conv = nn.Identity()
            self.out_channels = [16, 24, 32, 96, 320]

    def forward(self, x) -> List[torch.Tensor]:
        x = self.stem(x)        # (B, 32, H/2, W/2)
        f1 = self.stage1(x)     # (B, 16, H/2, W/2)
        f2 = self.stage2(f1)    # (B, 24, H/4, W/4)
        f3 = self.stage3(f2)    # (B, 32, H/8, W/8)
        f4 = self.stage4(f3)    # (B, 96, H/16, W/16)
        f5 = self.stage5(f4)    # (B, 320, H/32 or H/16, W/32 or W/16)
        f5 = self.top_conv(f5)  # (B, 1280, ...) when include_top_conv=True
        return [f1, f2, f3, f4, f5]


### 1.1 Summary info of `MobileNetV2Encoder`

In [ ]:
from torchinfo import summary
# Let's inspect the MobileNetV2 encoder backbone
test_model = MobileNetV2Encoder(Cin=3)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## 2. The ASPP head

Five parallel branches over the (B, 320, 32, 32) backbone output (input 512):

```python
branch 1: 1x1 conv                        -> (B, 256, 32, 32)
branch 2: 3x3 conv, dilation 6            -> (B, 256, 32, 32)
branch 3: 3x3 conv, dilation 12           -> (B, 256, 32, 32)
branch 4: 3x3 conv, dilation 18           -> (B, 256, 32, 32)
branch 5: global avg pool + 1x1 + upsample-> (B, 256, 32, 32)
concat (B, 1280, 32, 32) -> 1x1 project + dropout -> (B, 256, 32, 32)
```


In [ ]:
class ASPP(nn.Module):
    '''
    Class that defines the Atrous Spatial Pyramid Pooling module: parallel
    1x1 and dilated 3x3 convolutions plus global image pooling, concatenated
    and projected to `n` channels.

    Args:
        m (int): number of input channels (320 for MobileNetV2).
        n (int): number of output channels (256 in the paper).
        rates (tuple): dilation rates of the atrous branches (output stride 16).
    '''

    def __init__(self, m: int = 320, n: int = 256,
                 rates: tuple = (6, 12, 18)) -> None:
        super().__init__()

        # Branch 1: simple 1x1 convolution
        self.branch_1x1 = ConvBNReLU(m=m, n=n, kernel_size=1, stride=1, padding=0)

        # Branches 2-4: 3x3 atrous convolutions with increasing rates
        self.branch_atrous = nn.ModuleList([
            ConvBNReLU(m=m, n=n, kernel_size=3, stride=1,
                       padding=rate, dilation=rate)
            for rate in rates
        ])

        # Branch 5: image-level global average pooling
        self.branch_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=1),
            ConvBNReLU(m=m, n=n, kernel_size=1, stride=1, padding=0)
        )

        # Projection of the concatenated branches with dropout
        self.project = nn.Sequential(
            ConvBNReLU(m=n * (2 + len(rates)), n=n, kernel_size=1, stride=1, padding=0),
            nn.Dropout2d(p=0.5)
        )

    def forward(self, x) -> torch.Tensor:
        size = x.shape[2:]
        branches = [self.branch_1x1(x)]
        for branch in self.branch_atrous:
            branches.append(branch(x))
        pooled = self.branch_pool(x)
        pooled = F.interpolate(pooled, size=size, mode="bilinear", align_corners=True)
        branches.append(pooled)
        return self.project(torch.cat(branches, dim=1))


## 3. Final step, let's create the entire network


In [ ]:
class DeepLabV3MobileNetV2Model(nn.Module):
    '''
    Class that defines the full DeepLabV3 architecture with a MobileNetV2
    backbone at output stride 16 (last stage dilated), an ASPP head, and a
    16x bilinear upsampling back to full resolution.

    Args:
        Cin (int): number of input channels for the encoder.
        N (int): number of output channels (1 binary / num_classes semantic).
        aspp_channels (int): number of ASPP output channels (default 256).
    '''

    def __init__(self, Cin: int = 3, N: int = 1, aspp_channels: int = 256) -> None:
        super().__init__()

        # Dilated backbone: final feature is (B, 320, H/16, W/16)
        self.encoder = MobileNetV2Encoder(Cin=Cin, output_stride=16)
        c5 = self.encoder.out_channels[-1]

        # ASPP head with rates (6, 12, 18) for output stride 16
        self.aspp = ASPP(m=c5, n=aspp_channels, rates=(6, 12, 18))

        # Classifier: one 3x3 refinement conv + 1x1 projection to N logits
        self.head_conv = ConvBNReLU(m=aspp_channels, n=aspp_channels,
                                    kernel_size=3, stride=1, padding=1)
        self.segmentation_head = nn.Conv2d(in_channels=aspp_channels,
                                           out_channels=N, kernel_size=(1, 1),
                                           stride=(1, 1), padding=(0, 0))

    def forward(self, x) -> torch.Tensor:
        features = self.encoder(x)
        d = self.aspp(features[-1])          # (B, 256, H/16, W/16)
        logits = self.segmentation_head(self.head_conv(d))
        return F.interpolate(logits, size=x.shape[2:], mode="bilinear", align_corners=True)


### 3.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [ ]:
from torchinfo import summary
# Full DeepLabV3-MobileNetV2 model at 1024x1024
test_model = DeepLabV3MobileNetV2Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


### 3.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [ ]:
from torchinfo import summary
# Full DeepLabV3-MobileNetV2 model at 512x512
test_model = DeepLabV3MobileNetV2Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## Integration with `src/models` and the training framework

The exact same classes above live in `src/models`, and the `deeplabv3-mobilenetv2` model can be
built through the shared model factory. This is what the training scripts use when you pass
`--arch deeplabv3-mobilenetv2`:

```bash
python scripts/train_binary_text.py --arch deeplabv3-mobilenetv2
python scripts/train_semantic_layout.py --arch deeplabv3-mobilenetv2
```

Let's double check the factory produces the same model, and run the IEEE efficiency
benchmark (parameters, FLOPs, inference latency/FPS, and peak memory - GPU if available,
otherwise CPU RSS) with `src.utils.benchmark`.


In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root (src.*)
sys.path.insert(0, str(Path().cwd().parent))

from src.models import build_model

factory_model = build_model("deeplabv3-mobilenetv2", Cin=3, N=1)
total_params = sum(p.numel() for p in factory_model.parameters())
print(f"DeepLabV3-MobileNetV2 total parameters: {total_params:,} ({total_params/1e6:.2f}M)")


In [ ]:
from src.utils import benchmark_model, print_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
report = benchmark_model(
    model=factory_model,
    input_size=(1, 3, 512, 512),
    device=device,
    warmup=5,
    iterations=20,
    arch_name="deeplabv3-mobilenetv2",
)
print_benchmark(report)
